# MyDigitalTwin — Google & YouTube
**Notebook 06 — Ingestion, exploration, nettoyage → Parquet**

Sources :
- `Mon activité chez Google/Recherche/MonActivité.html` → recherches Google
- `Mon activité chez Google/Chrome/MonActivité.html` → historique Chrome
- `historique youtube (2023 - now)/watch-history.html` → vidéos YouTube regardées
- `historique youtube (2023 - now)/Historique des recherches.html` → recherches YouTube

Outputs :
- `data/parquet/google_searches.parquet`
- `data/parquet/google_chrome.parquet`
- `data/parquet/youtube_watch.parquet`
- `data/parquet/youtube_searches.parquet`

## Objectifs ML
- **NLP Clone (axe 1)** : requêtes Google → vocabulaire naturel
- **ALS (axe 2)** : YouTube watch → signal d'intérêt
- **K-Means (axe 3)** : activité temporelle toutes sources

## 0. Initialisation

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from bs4 import BeautifulSoup
from datetime import datetime, timezone
import re

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Google") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

GOOGLE_ROOT = "../../data/raw/GOOGLE"
PARQUET_DIR = "../../data/parquet"

# Mapping mois français → numéro
MOIS = {
    "janv": 1, "févr": 2, "mars": 3, "avr": 4, "mai": 5, "juin": 6,
    "juil": 7, "août": 8, "sept": 9, "oct": 10, "nov": 11, "déc": 12
}

def parse_google_date(text):
    """
    Parse les dates Google Takeout en français.
    Ex: '15 mars 2026, 18:29:40 CET'
        '20 mars 2026, 08:24:55 CET'
    """
    if not text:
        return 0
    m = re.search(
        r'(\d{1,2})\s+(\w+\.?)\s+(\d{4}),?\s+(\d{2}):(\d{2}):(\d{2})',
        text
    )
    if not m:
        return 0
    day, month_str, year, h, mi, s = m.groups()
    month_key = month_str.lower().rstrip('.')
    month_num = MOIS.get(month_key[:4], None)
    if not month_num:
        return 0
    try:
        dt = datetime(int(year), month_num, int(day), int(h), int(mi), int(s),
                      tzinfo=timezone.utc)
        return int(dt.timestamp() * 1000)
    except:
        return 0

def parse_html_activity(html_path, activity_type):
    """
    Parser générique pour les fichiers MonActivité.html Google.
    Retourne une liste de dicts avec : title, url, timestamp_ms, activity_type
    """
    print(f"  Lecture : {html_path.split('/')[-1]}...")
    with open(html_path, encoding="utf-8", errors="replace") as f:
        content = f.read()

    soup  = BeautifulSoup(content, "lxml")
    cells = soup.find_all("div", class_="outer-cell")
    print(f"  Cellules trouvées : {len(cells):,}")

    rows = []
    for cell in cells:
        content_div = cell.find("div", class_="content-cell")
        if not content_div:
            continue

        links    = content_div.find_all("a")
        main_lnk = links[0] if links else None
        sec_lnk  = links[1] if len(links) > 1 else None

        title  = main_lnk.get_text(strip=True) if main_lnk else ""
        url    = main_lnk.get("href", "")[:300] if main_lnk else ""
        author = sec_lnk.get_text(strip=True) if sec_lnk and \
                 any(x in sec_lnk.get("href", "") for x in ["channel", "user", "@"]) else ""

        full_text  = content_div.get_text(separator=" | ")
        ts_ms      = parse_google_date(full_text)

        if not title and not ts_ms:
            continue

        rows.append({
            "title":         title,
            "url":           url,
            "author":        author,
            "timestamp_ms":  ts_ms,
            "activity_type": activity_type,
            "platform":      "google",
        })

    print(f"  Lignes extraites : {len(rows):,}")
    return rows

def make_df(rows, extra_fields=None):
    """Crée un DataFrame PySpark avec champs temporels."""
    schema = StructType([
        StructField("title",         StringType(), True),
        StructField("url",           StringType(), True),
        StructField("author",        StringType(), True),
        StructField("timestamp_ms",  LongType(),   True),
        StructField("activity_type", StringType(), True),
        StructField("platform",      StringType(), True),
    ])
    df = spark.createDataFrame(rows, schema=schema)
    df = df \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("char_count",    F.length("title")) \
        .withColumn("word_count",    F.size(F.split(F.trim("title"), r"\s+")))
    return df

print("✓ Fonctions prêtes")

Spark version : 3.5.5
✓ Fonctions prêtes


26/03/28 14:48:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE 1 — Recherches Google
### 1.1 Ingestion

In [2]:
search_path = f"{GOOGLE_ROOT}/Mon activité chez Google/Recherche/MonActivité.html"

search_rows = parse_html_activity(search_path, "google_search")
df_search   = make_df(search_rows)

# Extraire le terme de recherche depuis l'URL
# Ex: https://www.google.fr/search?q=afterwork+bruxelles
df_search = df_search.withColumn(
    "query",
    F.regexp_extract(F.col("url"), r"[?&]q=([^&]+)", 1)
).withColumn(
    "query",
    F.regexp_replace(F.col("query"), r"\+", " ")
)

print(f"\nDataFrame : {df_search.count():,} lignes")
df_search.select("query", "title", "event_date").show(10, truncate=60)

  Lecture : MonActivité.html...
  Cellules trouvées : 55,854
  Lignes extraites : 55,854



DataFrame : 55,854 lignes
+------------------------------------------------------------+------------------------------------------------------------+-------------------+
|                                                       query|                                                       title|         event_date|
+------------------------------------------------------------+------------------------------------------------------------+-------------------+
|                                        http://lotusbleu.be/|                                        http://lotusbleu.be/|2026-03-15 18:29:40|
|streamlit.errors.StreamlitSecretNotFoundError: No secrets...|streamlit.errors.StreamlitSecretNotFoundError: No secrets...|2026-03-05 13:37:43|
|https://stackoverflow.com/questions/72112754/importerror-...|python - ImportError "no pq wrapper available" when impor...|2026-02-18 17:32:13|
|ImportError: no pq wrapper available.%0AAttempts made:%0A...|ImportError: no pq wrapper available.\nAttempts

### 1.2 Exploration

In [3]:
print("=== Recherches par année ===")
df_search.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_search.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Top 20 mots recherchés ===")
df_search.filter(F.col("query") != "") \
    .withColumn("word", F.explode(F.split(F.lower("query"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

print("\n=== Top 20 requêtes complètes ===")
df_search.filter(F.col("query") != "") \
    .groupBy("query").count() \
    .orderBy(F.desc("count")).limit(20).show(truncate=50)

=== Recherches par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2016|  134|
|      2017|  104|
|      2018| 1427|
|      2019| 5549|
|      2020| 5050|
|      2021| 8066|
|      2022|10369|
|      2023| 9612|
|      2024|10803|
|      2025| 4667|
|      2026|   73|
+----------+-----+


=== Activité par heure ===


+----------+-----+
|event_hour|count|
+----------+-----+
|         0| 1649|
|         1|  553|
|         2|  315|
|         3|   79|
|         4|   28|
|         5|  167|
|         6|  342|
|         7|  955|
|         8| 1552|
|         9| 2040|
|        10| 2366|
|        11| 2725|
|        12| 2544|
|        13| 3133|
|        14| 3426|
|        15| 3169|
|        16| 3787|
|        17| 4061|
|        18| 3965|
|        19| 3221|
+----------+-----+
only showing top 20 rows


=== Top 20 mots recherchés ===


+---------+-----+
|     word|count|
+---------+-----+
|      des|  357|
|      les|  355|
| belgique|  312|
|      une|  292|
|    faire|  286|
|      sur|  270|
|     fifa|  266|
|  youtube|  265|
|      one|  261|
|   google|  242|
|streaming|  227|
|     pour|  216|
|    piece|  203|
|minecraft|  203|
|      the|  197|
|   %c3%a0|  189|
|  comment|  184|
|     plus|  183|
|     avec|  181|
|     prix|  177|
+---------+-----+


=== Top 20 requêtes complètes ===


+--------------------------------+-----+
|                           query|count|
+--------------------------------+-----+
|                         youtube|   89|
|https://www.dreamland.be/e/fr/dl|   70|
|                          amazon|   54|
|                            hdss|   49|
|                            sncb|   44|
|                          krefel|   40|
|      https://tickets.imagix.be/|   39|
|                   google flight|   36|
|                       dreamland|   33|
|                        stg gege|   33|
|                             hds|   33|
|                https://hdss.to/|   32|
|                         11 anim|   31|
|          https://www.amazon.fr/|   30|
|                           apple|   28|
|                             tec|   28|
| http://www.film-streaming.club/|   28|
|                         zalando|   28|
|  https://www.belgiantrain.be/fr|   27|
|                          imagix|   27|
+--------------------------------+-----+



### 1.3 Écriture Parquet

In [4]:
df_search.write.mode("overwrite").parquet(f"{PARQUET_DIR}/google_searches.parquet")
print(f"✓ google_searches.parquet — {df_search.count():,} lignes")

✓ google_searches.parquet — 55,854 lignes


---
## PARTIE 2 — Historique Chrome
### 2.1 Ingestion

In [5]:
chrome_path = f"{GOOGLE_ROOT}/Mon activité chez Google/Chrome/MonActivité.html"

chrome_rows = parse_html_activity(chrome_path, "chrome_visit")
df_chrome   = make_df(chrome_rows)

# Extraire le domaine visité
df_chrome = df_chrome.withColumn(
    "domain",
    F.regexp_extract(F.col("url"), r"https?://(?:www\.)?([^/]+)", 1)
)

print(f"\nDataFrame : {df_chrome.count():,} lignes")
df_chrome.select("title", "domain", "event_date").show(10, truncate=60)

  Lecture : MonActivité.html...
  Cellules trouvées : 338
  Lignes extraites : 338

DataFrame : 338 lignes
+------------------------------+----------+-------------------+
|                         title|    domain|         event_date|
+------------------------------+----------+-------------------+
|                              |          |2025-12-29 22:31:38|
|Photopea | Online Photo Editor|google.com|2025-12-23 23:13:40|
|Photopea | Online Photo Editor|google.com|2025-12-13 20:37:36|
|Photopea | Online Photo Editor|google.com|2025-12-10 21:36:30|
|Photopea | Online Photo Editor|google.com|2025-11-30 23:50:54|
|Photopea | Online Photo Editor|google.com|2025-11-30 21:35:34|
|Photopea | Online Photo Editor|google.com|2025-11-27 23:13:46|
|Photopea | Online Photo Editor|google.com|2025-11-15 19:50:27|
|Photopea | Online Photo Editor|google.com|2025-11-07 17:01:31|
|Photopea | Online Photo Editor|google.com|2025-11-07 13:30:42|
+------------------------------+----------+------------------

### 2.2 Exploration

In [6]:
print("=== Visites par année ===")
df_chrome.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Top 20 domaines visités ===")
df_chrome.filter(F.col("domain") != "") \
    .groupBy("domain").count() \
    .orderBy(F.desc("count")).limit(20).show(truncate=50)

print("\n=== Activité par heure ===")
df_chrome.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_hour").count().orderBy("event_hour").show()

=== Visites par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2025|  338|
+----------+-----+


=== Top 20 domaines visités ===
+-------------------------+-----+
|                   domain|count|
+-------------------------+-----+
|               google.com|  294|
|          mail.google.com|   24|
|                google.be|    8|
|              youtube.com|    5|
|chromewebstore.google.com|    3|
|      accounts.google.com|    2|
+-------------------------+-----+


=== Activité par heure ===


+----------+-----+
|event_hour|count|
+----------+-----+
|         0|    8|
|         1|    1|
|        10|    1|
|        12|   42|
|        13|    6|
|        15|    3|
|        16|    1|
|        17|    1|
|        18|    3|
|        19|    3|
|        20|   74|
|        21|   40|
|        22|   61|
|        23|   94|
+----------+-----+



### 2.3 Écriture Parquet

In [7]:
df_chrome.write.mode("overwrite").parquet(f"{PARQUET_DIR}/google_chrome.parquet")
print(f"✓ google_chrome.parquet — {df_chrome.count():,} lignes")

✓ google_chrome.parquet — 338 lignes


---
## PARTIE 3 — YouTube Watch History
### 3.1 Ingestion

In [2]:
yt_watch_path = f"{GOOGLE_ROOT}/historique youtube (2023 - now)/watch-history.html"

yt_watch_rows = parse_html_activity(yt_watch_path, "youtube_watch")
df_yt_watch   = make_df(yt_watch_rows)

# Extraire l'ID vidéo YouTube
df_yt_watch = df_yt_watch \
    .withColumn(
        "video_id",
        F.regexp_extract(F.col("url"), r"[?&]v=([a-zA-Z0-9_-]{11})", 1)
    ) \
    .withColumn("interaction_weight", F.lit(1.0))

print(f"\nDataFrame : {df_yt_watch.count():,} lignes")
df_yt_watch.select("title", "author", "video_id", "event_date").show(10, truncate=50)

  Lecture : watch-history.html...
  Cellules trouvées : 14,103
  Lignes extraites : 14,103



DataFrame : 14,103 lignes
+--------------------------------------------------+-----------------------+-----------+-------------------+
|                                             title|                 author|   video_id|         event_date|
+--------------------------------------------------+-----------------------+-----------+-------------------+
|                          MIT Maker Portfolio 2028|             AstroSam 2|QF2l-0Vb2VM|2026-03-20 08:24:55|
|Big Data Analytics | What Is Big Data Analytics...|            Simplilearn|bY6ZzQmtOzk|2026-03-20 08:20:41|
|Projet Big Data :  Maîtriser l’intégration des ...|         Jaweher Hichri|BOPmswG3Kog|2026-03-20 08:20:09|
|                       🌐 Le Big Data c'est QUOI ?|             Enissay ✔️|9x7hVEyynbA|2026-03-20 08:19:58|
|Minecraft : 4 Nullos, 1 Barre de vie - Rediffus...|Squeezie - Rediffusions|b6RWy6szb5E|2026-03-19 21:14:40|
|Au vu de l'engouement, on a déjà rempli la plus...|                 Micode|           |2026-03-19 20:

### Filtrage des pubs

In [3]:
# Filtrer les publicités
df_yt_watch = df_yt_watch.filter(
    # Exclure les URLs publicitaires
    ~F.col("url").contains("googleadservices") &
    ~F.col("url").contains("doubleclick") &
    ~F.col("url").contains("googlevideo.com/videoplayback") &
    # Exclure les titres génériques de pubs
    ~F.col("title").rlike(r"(?i)^(publicité|pub|advertisement|ad)$") &
    # Garder uniquement les vraies vidéos YouTube (avec video_id)
    (F.col("video_id") != "")
)

print(f"Après filtrage pubs : {df_yt_watch.count():,} lignes")

Après filtrage pubs : 13,821 lignes


### 3.2 Exploration

In [4]:
print("=== Vidéos regardées par année ===")
df_yt_watch.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Top 15 chaînes les plus regardées ===")
df_yt_watch.filter(F.col("author") != "") \
    .groupBy("author").count() \
    .orderBy(F.desc("count")).limit(15).show(truncate=50)

print("\n=== Top 15 vidéos les plus regardées ===")
df_yt_watch.filter(F.col("title") != "") \
    .groupBy("title", "author").count() \
    .orderBy(F.desc("count")).limit(15).show(truncate=50)

print("\n=== Activité par heure ===")
df_yt_watch.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Top 20 mots dans les titres ===")
df_yt_watch.filter(F.col("title") != "") \
    .withColumn("word", F.explode(F.split(F.lower("title"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

=== Vidéos regardées par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2023| 2531|
|      2024| 4593|
|      2025| 5533|
|      2026| 1164|
+----------+-----+


=== Top 15 chaînes les plus regardées ===


+-----------------------+-----+
|                 author|count|
+-----------------------+-----+
|               SQUEEZIE|  163|
|             Squa' Sh*t|  144|
|Squeezie - Rediffusions|  124|
|                  AlaGT|   77|
|           Zen Emission|   73|
|        SQUEEZIE GAMING|   67|
|               Anyme TV|   65|
|                Arsenal|   65|
|                  HOUDI|   63|
|                  Mastu|   62|
|          Lacrimosophia|   56|
|          DJ ZAYON PROD|   53|
|Locklear Replays & VODs|   50|
|        TravisScottVEVO|   45|
|                Inoxtag|   44|
+-----------------------+-----+


=== Top 15 vidéos les plus regardées ===


+--------------------------------------------------+----------+-----+
|                                             title|    author|count|
+--------------------------------------------------+----------+-----+
|                                 Choisissez Chrome|          |  114|
| Changez d'avis sur le reconditionné - Back Market|          |   27|
|                BNPPF_HY4 Boost_9x16_FR_12sec_2505|          |   25|
|INH INH Budget 329918 Linkedaccounts VID 16x9 6...|          |   20|
|                             CFC英语口播 0202 横版|          |   19|
|    Een lening om mijn zoon in Sydney te bezoeken?|          |   19|
|                                       Kies Chrome|          |   18|
|DEWALT® UK | On the jobsite, leaders aren't bor...|          |   17|
|INH INH Retail 572 FreeBankAccount VID 16x9 10s...|          |   17|
|                                    HOUDI - MONACO|     HOUDI|   17|
|                       Gazo - Guinea Man [Mixtape]|Squa' Sh*t|   16|
|Enjoy a better CI/CD expe

+----------+-----+
|event_hour|count|
+----------+-----+
|         0|  500|
|         1|  296|
|         2|  179|
|         3|   80|
|         4|   10|
|         5|   25|
|         6|   46|
|         7|  124|
|         8|  185|
|         9|  313|
|        10|  309|
|        11|  547|
|        12|  670|
|        13|  889|
|        14|  846|
|        15|  763|
|        16|  823|
|        17|  772|
|        18|  992|
|        19|  988|
+----------+-----+
only showing top 20 rows


=== Top 20 mots dans les titres ===


+---------+-----+
|     word|count|
+---------+-----+
|      the|  878|
|      les|  577|
|     16x9|  571|
|      inh|  508|
|      how|  389|
|      des|  386|
|     pour|  318|
|     avec|  318|
|     with|  311|
|officiel)|  310|
|    (clip|  309|
|      for|  265|
|      mix|  256|
|      and|  251|
|      une|  242|
|(official|  241|
|      qui|  235|
|   video)|  234|
|     your|  228|
|      you|  226|
+---------+-----+



### 3.3 Écriture Parquet

In [5]:
df_yt_watch.write.mode("overwrite").parquet(f"{PARQUET_DIR}/youtube_watch.parquet")
print(f"✓ youtube_watch.parquet — {df_yt_watch.count():,} lignes")

✓ youtube_watch.parquet — 13,821 lignes


---
## PARTIE 4 — Recherches YouTube
### 4.1 Ingestion

In [6]:
yt_search_path = f"{GOOGLE_ROOT}/historique youtube (2023 - now)/Historique des recherches.html"

yt_search_rows = parse_html_activity(yt_search_path, "youtube_search")
df_yt_search   = make_df(yt_search_rows)

print(f"\nDataFrame : {df_yt_search.count():,} lignes")
df_yt_search.select("title", "event_date").show(10, truncate=60)

  Lecture : Historique des recherches.html...
  Cellules trouvées : 5,835
  Lignes extraites : 5,835



DataFrame : 5,835 lignes
+------------------------------------------------------------+-------------------+
|                                                       title|         event_date|
+------------------------------------------------------------+-------------------+
|                             big data outil de développement|2026-03-20 08:20:35|
|                                                    big data|2026-03-20 08:18:56|
|                                        squeezie rediffusion|2026-03-19 17:54:14|
|Shortened: ROK_V_EN_ZRSP_AirlineStepIn_CB_1_1920x1080_58s...|2026-03-18 14:00:27|
|Shortened: RAD AliceChesire 30sec V227284 30 1920x1080 S6...|2026-03-18 14:00:20|
|Made For More Adventures - E-Adventure Bikes - E+ System ...|2026-03-18 13:56:45|
|         Shortened: GoogleAC_English(GB)_AYF-Choice_9x16_20s|2026-03-18 13:56:38|
|                  Peaky Blinders: The Immortal Man | Netflix|2026-03-18 13:49:28|
|INH PARTNERSHIP F1 793c8dca CarCardSocialProof LS 16x9 6s...

In [7]:
df_yt_search = df_yt_search.filter(
    # Exclure les titres de pubs (patterns reconnaissables)
    ~F.col("title").rlike(r"(?i)^Shortened:") &
    ~F.col("title").rlike(r"(?i)^(RAD|ROK|INH|AYF|ADS|ADV)[\s_]") &
    ~F.col("title").rlike(r"[0-9]{4}x[0-9]{3,4}") &  # dimensions vidéo ex: 1920x1080
    ~F.col("title").rlike(r"_[0-9]+s$") &              # durée en secondes ex: _30s
    ~F.col("title").rlike(r"(?i)(partnership|campaign|creative)") &
    # Garder uniquement les vraies recherches (au moins 2 mots ou titre court naturel)
    (F.length("title") > 2) &
    ~F.col("title").rlike(r"^[A-Z0-9_\s]{20,}$")  # codes en majuscules = pubs
)

print(f"Après filtrage pubs : {df_yt_search.count():,} lignes")
df_yt_search.select("title", "event_date").show(10, truncate=60)

Après filtrage pubs : 4,991 lignes
+------------------------------------------------------------+-------------------+
|                                                       title|         event_date|
+------------------------------------------------------------+-------------------+
|                             big data outil de développement|2026-03-20 08:20:35|
|                                                    big data|2026-03-20 08:18:56|
|                                        squeezie rediffusion|2026-03-19 17:54:14|
|Made For More Adventures - E-Adventure Bikes - E+ System ...|2026-03-18 13:56:45|
|                  Peaky Blinders: The Immortal Man | Netflix|2026-03-18 13:49:28|
|                                  Économisez €245 en moyenne|2026-03-18 13:42:18|
|                                 TheShark 15s Sales BENL 169|2026-03-18 13:34:53|
|Goûte l'Apple Caramel de Jonatan. Si sucré, ça devrait êt...|2026-03-18 13:34:47|
|                                        DSM Zavente

### 4.2 Exploration

In [8]:
print("=== Recherches YouTube par année ===")
df_yt_search.filter(F.col("timestamp_ms") > 0) \
    .groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Top 20 termes recherchés ===")
df_yt_search.groupBy("title").count() \
    .orderBy(F.desc("count")).limit(20).show(truncate=60)

print("\n=== Top 20 mots ===")
df_yt_search.filter(F.col("title") != "") \
    .withColumn("word", F.explode(F.split(F.lower("title"), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .groupBy("word").count() \
    .orderBy(F.desc("count")).limit(20).show()

=== Recherches YouTube par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2023|  765|
|      2024| 1753|
|      2025| 2146|
|      2026|  327|
+----------+-----+


=== Top 20 termes recherchés ===


+------------------------------------------------------------+-----+
|                                                       title|count|
+------------------------------------------------------------+-----+
|                                           Choisissez Chrome|   39|
|                          BNPPF_HY4 Boost_9x16_FR_12sec_2505|   26|
|              Een lening om mijn zoon in Sydney te bezoeken?|   19|
|DEWALT® UK | On the jobsite, leaders aren't born. They're...|   18|
|                                       CFC英语口播 0202 横版|   18|
|                                   Le pack ING Do More 18-25|   11|
|FINAL 15123076 FY23Q4 PsMax PerformanceMedia DistractorRe...|   11|
|           Changez d'avis sur le reconditionné - Back Market|   11|
|                             DSP BE FR | Beer With LatinVibe|   10|
|                                                 Kies Chrome|   10|
|                          Nouveau! Déodorant Axe Cherry Fizz|   10|
|                                       

+------+-----+
|  word|count|
+------+-----+
|  16x9|  229|
|   the|  141|
|  befr|   97|
|   15s|   92|
|  with|   91|
|   les|   85|
|   des|   75|
|  pour|   73|
|  your|   58|
| video|   58|
|  2024|   57|
|  avec|   55|
| 20sec|   53|
|   for|   52|
|  2025|   52|
|   sur|   50|
|   how|   50|
|chrome|   49|
|seller|   47|
|   sec|   44|
+------+-----+



### 4.3 Écriture Parquet

In [9]:
df_yt_search.write.mode("overwrite").parquet(f"{PARQUET_DIR}/youtube_searches.parquet")
print(f"✓ youtube_searches.parquet — {df_yt_search.count():,} lignes")

✓ youtube_searches.parquet — 4,991 lignes


---
## Résumé

In [ ]:
print("=" * 60)
print("  MyDigitalTwin — Google & YouTube — Résumé")
print("=" * 60)
print(f"  Recherches Google : {df_search.count():>8,} → google_searches.parquet")
print(f"  Chrome            : {df_chrome.count():>8,} → google_chrome.parquet")
print(f"  YouTube watch     : {df_yt_watch.count():>8,} → youtube_watch.parquet")
print(f"  YouTube recherches: {df_yt_search.count():>8,} → youtube_searches.parquet")
total = df_search.count() + df_chrome.count() + df_yt_watch.count() + df_yt_search.count()
print(f"  {'─'*45}")
print(f"  TOTAL             : {total:>8,} lignes")
print("=" * 60)
print()
print("  Usage ML :")
print("  - Recherches Google → NLP Clone (axe 1)")
print("  - YouTube watch     → ALS interactions (axe 2, weight=1)")
print("  - Toutes sources    → K-Means temporel (axe 3)")

In [10]:
spark.stop()